<a href="https://colab.research.google.com/github/MOHAMMEDAWEZALI2303A51767/RL_Lab/blob/main/Lab_Assign_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

# ==========================================
# Task 1: Inspect the MDP & Define Parameters
# ==========================================
print("=== Task 1: MDP Parameters & Configuration ===")
states = ['S0', 'S1']
terminal_state = 'G'
actions = ['Up', 'Down', 'Left', 'Right']

gamma = 0.9          # Discount factor
theta = 1e-6         # Convergence threshold

print(f"Non-terminal States: {states}")
print(f"Terminal State: {terminal_state}")
print(f"Actions: {actions}")
print(f"Discount Factor (gamma): {gamma}")
print(f"Theta (convergence threshold): {theta}\n")

# Define Transition Function: P(s' | s, a) and Rewards R(s, a, s')
# Based on the 2x2 grid world setup:
# - Moving into normal cell or hitting wall yields reward = -1
# - Reaching goal cell 'G' yields reward = +10
def get_transition(state, action):
    if state == 'S0':
        if action == 'Up':    return 'S0', -1
        if action == 'Down':  return 'G', 10
        if action == 'Left':  return 'S0', -1
        if action == 'Right': return 'S1', -1

    elif state == 'S1':
        if action == 'Up':    return 'S1', -1
        if action == 'Down':  return 'G', 10
        if action == 'Left':  return 'S0', -1
        if action == 'Right': return 'S1', -1

    return state, 0

# ==========================================
# Task 2: Verify Transitions
# ==========================================
print("=== Task 2: Verify Transitions ===")
transition_records = []
for s in states:
    for a in actions:
        next_s, r = get_transition(s, a)
        transition_records.append({'State': s, 'Action': a, 'Next State': next_s, 'Reward': r})
        print(f"State: {s} | Action: {a} --> Next State: {next_s} | Reward: {r}")

trans_df = pd.DataFrame(transition_records)
display(trans_df)

# ==========================================
# Task 3: Initialize State Values
# ==========================================
print("\n=== Task 3: Initialize State Values ===")
V = {'S0': 0.0, 'S1': 0.0, 'G': 0.0}
print("Initial State Values V(s):", V)

# ==========================================
# Task 4 & 5: Implement Value Iteration & Convergence Log
# ==========================================
print("\n=== Tasks 4 & 5: Value Iteration Convergence Log ===")
iteration = 0
history = []

while True:
    delta = 0
    V_new = V.copy()

    for s in states:
        q_values = []
        for a in actions:
            next_s, r = get_transition(s, a)
            # Bellman Optimality Equation: V_{k+1}(s) = max_a [R + gamma * V_k(s')]
            q = r + gamma * V[next_s]
            q_values.append(q)

        V_new[s] = max(q_values)
        change = abs(V_new[s] - V[s])
        delta = max(delta, change)

    history.append({
        'Iteration': iteration,
        'V(S0)': V_new['S0'],
        'V(S1)': V_new['S1'],
        'Delta (Δ)': delta
    })

    print(f"Iteration {iteration}: V(S0) = {V_new['S0']:.4f}, V(S1) = {V_new['S1']:.4f}, Delta = {delta:.6f}")

    V = V_new
    iteration += 1

    if delta < theta:
        print(f"\nConverged successfully in {iteration} iterations!")
        break

history_df = pd.DataFrame(history)
display(history_df)

# ==========================================
# Task 6: Calculate Action Values Q*(s, a)
# ==========================================
print("\n=== Task 6: Final Action Values Q*(s, a) ===")
Q_table = []
for s in states:
    for a in actions:
        next_s, r = get_transition(s, a)
        q_val = r + gamma * V[next_s]
        Q_table.append({'State': s, 'Action': a, 'Q*(s,a)': q_val})
        print(f"Q*({s}, {a}) = {q_val:.4f}")

q_df = pd.DataFrame(Q_table)
display(q_df)

# ==========================================
# Task 7: Extract the Optimal Policy
# ==========================================
print("\n=== Task 7: Extract Optimal Policy ===")
optimal_policy = {}
for s in states:
    state_actions = q_df[q_df['State'] == s]
    max_q = state_actions['Q*(s,a)'].max()
    best_actions = state_actions[state_actions['Q*(s,a)'] == max_q]['Action'].tolist()

    optimal_policy[s] = best_actions
    print(f"Optimal Policy for {s}: {best_actions} (Max Q = {max_q:.4f})")

# ==========================================
# Task 8: Verify the Result
# ==========================================
print("\n=== Task 8: Bellman Optimality Verification ===")
verification_records = []
for s in states:
    v_s = V[s]
    max_q = q_df[q_df['State'] == s]['Q*(s,a)'].max()
    match = np.isclose(v_s, max_q)
    verification_records.append({'State': s, 'V*(s)': v_s, 'max_a Q*(s,a)': max_q, 'Verified Match': match})
    print(f"State {s}: V*(s) = {v_s:.4f} | max_a Q*(s,a) = {max_q:.4f} | Match: {match}")

ver_df = pd.DataFrame(verification_records)
display(ver_df)

=== Task 1: MDP Parameters & Configuration ===
Non-terminal States: ['S0', 'S1']
Terminal State: G
Actions: ['Up', 'Down', 'Left', 'Right']
Discount Factor (gamma): 0.9
Theta (convergence threshold): 1e-06

=== Task 2: Verify Transitions ===
State: S0 | Action: Up --> Next State: S0 | Reward: -1
State: S0 | Action: Down --> Next State: G | Reward: 10
State: S0 | Action: Left --> Next State: S0 | Reward: -1
State: S0 | Action: Right --> Next State: S1 | Reward: -1
State: S1 | Action: Up --> Next State: S1 | Reward: -1
State: S1 | Action: Down --> Next State: G | Reward: 10
State: S1 | Action: Left --> Next State: S0 | Reward: -1
State: S1 | Action: Right --> Next State: S1 | Reward: -1


,State,Action,Next State,Reward
0,S0,Up,S0,-1
1,S0,Down,G,10
2,S0,Left,S0,-1
3,S0,Right,S1,-1
4,S1,Up,S1,-1
5,S1,Down,G,10
6,S1,Left,S0,-1
7,S1,Right,S1,-1



=== Task 3: Initialize State Values ===
Initial State Values V(s): {'S0': 0.0, 'S1': 0.0, 'G': 0.0}

=== Tasks 4 & 5: Value Iteration Convergence Log ===
Iteration 0: V(S0) = 10.0000, V(S1) = 10.0000, Delta = 10.000000
Iteration 1: V(S0) = 10.0000, V(S1) = 10.0000, Delta = 0.000000

Converged successfully in 2 iterations!


,Iteration,V(S0),V(S1),Delta (Δ)
0,0,10.0,10.0,10.0
1,1,10.0,10.0,0.0



=== Task 6: Final Action Values Q*(s, a) ===
Q*(S0, Up) = 8.0000
Q*(S0, Down) = 10.0000
Q*(S0, Left) = 8.0000
Q*(S0, Right) = 8.0000
Q*(S1, Up) = 8.0000
Q*(S1, Down) = 10.0000
Q*(S1, Left) = 8.0000
Q*(S1, Right) = 8.0000


,State,Action,"Q*(s,a)"
0,S0,Up,8.0
1,S0,Down,10.0
2,S0,Left,8.0
3,S0,Right,8.0
4,S1,Up,8.0
5,S1,Down,10.0
6,S1,Left,8.0
7,S1,Right,8.0



=== Task 7: Extract Optimal Policy ===
Optimal Policy for S0: ['Down'] (Max Q = 10.0000)
Optimal Policy for S1: ['Down'] (Max Q = 10.0000)

=== Task 8: Bellman Optimality Verification ===
State S0: V*(s) = 10.0000 | max_a Q*(s,a) = 10.0000 | Match: True
State S1: V*(s) = 10.0000 | max_a Q*(s,a) = 10.0000 | Match: True


,State,V*(s),"max_a Q*(s,a)",Verified Match
0,S0,10.0,10.0,True
1,S1,10.0,10.0,True
